<a href="https://colab.research.google.com/github/falloudev190/Crous-T_local/blob/main/Projet_BDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
print("Bonjour le monde !")

Bonjour le monde !


In [ ]:
import sqlite3

def setup_crous_database():
    conn = sqlite3.connect('crous_vcn.db')
    cursor = conn.cursor()

    # Execution du schema valide sur SQLite Online
    sql_script = """
    PRAGMA foreign_keys = ON;

    CREATE TABLE IF NOT EXISTS utilisateurs (
        id_utilisateur INTEGER PRIMARY KEY AUTOINCREMENT,
        nom TEXT NOT NULL,
        prenom TEXT NOT NULL,
        email TEXT UNIQUE NOT NULL,
        telephone TEXT,
        role TEXT NOT NULL CHECK(role IN ('DEMANDEUR', 'AGENT', 'TECHNICIEN', 'SUPER_ADMIN'))
    );

    CREATE TABLE IF NOT EXISTS demandeurs (
        id_utilisateur INTEGER PRIMARY KEY,
        cni TEXT NOT NULL,
        est_etudiant BOOLEAN NOT NULL DEFAULT 0,
        num_etudiant TEXT,
        FOREIGN KEY (id_utilisateur) REFERENCES utilisateurs(id_utilisateur) ON DELETE CASCADE
    );

    CREATE TABLE IF NOT EXISTS agents (
        id_utilisateur INTEGER PRIMARY KEY,
        service TEXT NOT NULL CHECK(service IN ('COMMISSION', 'FINANCE', 'JURIDIQUE', 'DCUVE')),
        matricule TEXT UNIQUE NOT NULL,
        FOREIGN KEY (id_utilisateur) REFERENCES utilisateurs(id_utilisateur) ON DELETE CASCADE
    );

    CREATE TABLE IF NOT EXISTS techniciens (
        id_utilisateur INTEGER PRIMARY KEY,
        specialite TEXT NOT NULL,
        disponibilite BOOLEAN DEFAULT 1,
        FOREIGN KEY (id_utilisateur) REFERENCES utilisateurs(id_utilisateur) ON DELETE CASCADE
    );

    CREATE TABLE IF NOT EXISTS locaux_commerciaux (
        id_local INTEGER PRIMARY KEY AUTOINCREMENT,
        code_local TEXT UNIQUE NOT NULL,
        type_activite TEXT NOT NULL,
        surface_m2 REAL NOT NULL,
        statut TEXT DEFAULT 'Disponible' CHECK(statut IN ('Disponible', 'Occupé', 'En attente', 'Maintenance'))
    );

    CREATE TABLE IF NOT EXISTS demandes_local (
        id_demande INTEGER PRIMARY KEY AUTOINCREMENT,
        code_demande TEXT UNIQUE NOT NULL,
        type_demande TEXT NOT NULL CHECK(type_demande IN ('OBTENTION', 'CONSTRUCTION')),
        date_depot DATETIME DEFAULT CURRENT_TIMESTAMP,
        statut TEXT DEFAULT 'EN_ATTENTE' CHECK(statut IN ('EN_ATTENTE', 'EN_COURS', 'VALIDEE', 'REJETEE')),
        id_demandeur INTEGER NOT NULL,
        FOREIGN KEY (id_demandeur) REFERENCES demandeurs(id_utilisateur)
    );

    CREATE TABLE IF NOT EXISTS contrats (
        id_contrat INTEGER PRIMARY KEY AUTOINCREMENT,
        num_contrat TEXT UNIQUE NOT NULL,
        date_signature DATE,
        montant_loyer_mensuel REAL NOT NULL CHECK(montant_loyer_mensuel >= 0),
        valide_par_directeur BOOLEAN DEFAULT 0,
        statut TEXT DEFAULT 'EN_ATTENTE' CHECK(statut IN ('ACTIF', 'RESILIE', 'EN_ATTENTE')),
        id_demande INTEGER UNIQUE,
        id_local INTEGER NOT NULL,
        FOREIGN KEY (id_demande) REFERENCES demandes_local(id_demande),
        FOREIGN KEY (id_local) REFERENCES locaux_commerciaux(id_local)
    );

    CREATE TABLE IF NOT EXISTS paiements (
        id_paiement INTEGER PRIMARY KEY AUTOINCREMENT,
        montant REAL NOT NULL CHECK(montant > 0),
        date_paiement DATETIME DEFAULT CURRENT_TIMESTAMP,
        mode_paiement TEXT NOT NULL,
        id_contrat INTEGER NOT NULL,
        FOREIGN KEY (id_contrat) REFERENCES contrats(id_contrat)
    );
    """
    cursor.executescript(sql_script)

    # Seeding des utilisateurs
    cursor.execute("SELECT COUNT(*) FROM utilisateurs")
    if cursor.fetchone()[0] == 0:
        cursor.execute("INSERT INTO utilisateurs (nom, prenom, email, telephone, role) VALUES ('Admin', 'Super', 'admin@crous.sn', '770000000', 'SUPER_ADMIN')")

        cursor.execute("INSERT INTO utilisateurs (nom, prenom, email, telephone, role) VALUES ('Faye', 'Awa', 'awa.faye@crous.sn', '771112233', 'AGENT')")
        u_comm = cursor.lastrowid
        cursor.execute("INSERT INTO agents (id_utilisateur, service, matricule) VALUES (?, 'COMMISSION', 'AGT-COMM-01')", (u_comm,))

        cursor.execute("INSERT INTO utilisateurs (nom, prenom, email, telephone, role) VALUES ('Sene', 'Ousmane', 'ousmane.sene@crous.sn', '774445566', 'AGENT')")
        u_fin = cursor.lastrowid
        cursor.execute("INSERT INTO agents (id_utilisateur, service, matricule) VALUES (?, 'FINANCE', 'AGT-FIN-01')", (u_fin,))

        cursor.execute("INSERT INTO utilisateurs (nom, prenom, email, telephone, role) VALUES ('Ndiaye', 'Modou', 'modou.ndiaye@crous.sn', '778889900', 'TECHNICIEN')")
        u_tech = cursor.lastrowid
        cursor.execute("INSERT INTO techniciens (id_utilisateur, specialite, disponibilite) VALUES (?, 'Plomberie & Électricité', 1)", (u_tech,))

    conn.commit()

    # Verification des tables generees
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = [row[0] for row in cursor.fetchall() if row[0] not in ('sqlite_sequence', 'demo')]
    conn.close()

    print(f" Base de données créée dans Colab avec les tables : {', '.join(tables)}")

setup_crous_database()
# Une fois cette cellule exécutée dans Colab, le fichier `crous_vcn.db` sera disponible localement dans votre environnement de travail avec l'ensemble des contraintes de clés étrangères et la hiérarchie des rôles.

 Base de données créée dans Colab avec les tables : utilisateurs, demandeurs, agents, techniciens, locaux_commerciaux, demandes_local, contrats, paiements


In [ ]:
import threading
import time
from IPython.display import HTML, display
from app import run

# 1. Démarrage du serveur Backend relié à crous_vcn.db
t = threading.Thread(target=run, kwargs={'port': 8000}, daemon=True)
t.start()
time.sleep(1)

# 2. Affichage interactif grand écran
display(HTML('''
<iframe src="http://localhost:8000" style="width:100%; height:900px; border:none; border-radius:14px; box-shadow:0 4px 20px rgba(0,0,0,0.15);"></iframe>
'''))

In [14]:
import json
import random
import sqlite3
from datetime import datetime
from http.server import HTTPServer, SimpleHTTPRequestHandler
from urllib.parse import urlparse

DB_FILE = 'crous_vcn.db'

def get_db():
    conn = sqlite3.connect(DB_FILE)
    conn.row_factory = sqlite3.Row
    return conn

def setup_crous_database():
    conn = get_db()
    cursor = conn.cursor()

    sql_script = """
    PRAGMA foreign_keys = ON;

    CREATE TABLE IF NOT EXISTS utilisateurs (
        id_utilisateur INTEGER PRIMARY KEY AUTOINCREMENT,
        nom TEXT NOT NULL,
        prenom TEXT NOT NULL,
        email TEXT UNIQUE NOT NULL,
        telephone TEXT,
        role TEXT NOT NULL CHECK(role IN ('DEMANDEUR', 'AGENT', 'TECHNICIEN', 'SUPER_ADMIN'))
    );

    CREATE TABLE IF NOT EXISTS demandeurs (
        id_utilisateur INTEGER PRIMARY KEY,
        cni TEXT NOT NULL,
        est_etudiant BOOLEAN NOT NULL DEFAULT 0,
        num_etudiant TEXT,
        FOREIGN KEY (id_utilisateur) REFERENCES utilisateurs(id_utilisateur) ON DELETE CASCADE
    );

    CREATE TABLE IF NOT EXISTS agents (
        id_utilisateur INTEGER PRIMARY KEY,
        service TEXT NOT NULL CHECK(service IN ('COMMISSION', 'FINANCE', 'JURIDIQUE', 'DCUVE')),
        matricule TEXT UNIQUE NOT NULL,
        FOREIGN KEY (id_utilisateur) REFERENCES utilisateurs(id_utilisateur) ON DELETE CASCADE
    );

    CREATE TABLE IF NOT EXISTS techniciens (
        id_utilisateur INTEGER PRIMARY KEY,
        specialite TEXT NOT NULL,
        disponibilite BOOLEAN DEFAULT 1,
        FOREIGN KEY (id_utilisateur) REFERENCES utilisateurs(id_utilisateur) ON DELETE CASCADE
    );

    CREATE TABLE IF NOT EXISTS locaux_commerciaux (
        id_local INTEGER PRIMARY KEY AUTOINCREMENT,
        code_local TEXT UNIQUE NOT NULL,
        type_activite TEXT NOT NULL,
        surface_m2 REAL NOT NULL,
        statut TEXT DEFAULT 'Disponible' CHECK(statut IN ('Disponible', 'Occupé', 'En attente', 'Maintenance'))
    );

    CREATE TABLE IF NOT EXISTS demandes_local (
        id_demande INTEGER PRIMARY KEY AUTOINCREMENT,
        code_demande TEXT UNIQUE NOT NULL,
        type_demande TEXT NOT NULL CHECK(type_demande IN ('OBTENTION', 'CONSTRUCTION')),
        date_depot DATETIME DEFAULT CURRENT_TIMESTAMP,
        statut TEXT DEFAULT 'EN_ATTENTE' CHECK(statut IN ('EN_ATTENTE', 'EN_COURS', 'VALIDEE', 'REJETEE')),
        id_demandeur INTEGER NOT NULL,
        id_local INTEGER,
        FOREIGN KEY (id_demandeur) REFERENCES demandeurs(id_utilisateur),
        FOREIGN KEY (id_local) REFERENCES locaux_commerciaux(id_local)
    );

    CREATE TABLE IF NOT EXISTS contrats (
        id_contrat INTEGER PRIMARY KEY AUTOINCREMENT,
        num_contrat TEXT UNIQUE NOT NULL,
        date_signature DATE,
        montant_loyer_mensuel REAL NOT NULL CHECK(montant_loyer_mensuel >= 0),
        valide_par_directeur BOOLEAN DEFAULT 0,
        statut TEXT DEFAULT 'EN_ATTENTE' CHECK(statut IN ('ACTIF', 'RESILIE', 'EN_ATTENTE')),
        id_demande INTEGER UNIQUE,
        id_local INTEGER NOT NULL,
        FOREIGN KEY (id_demande) REFERENCES demandes_local(id_demande),
        FOREIGN KEY (id_local) REFERENCES locaux_commerciaux(id_local)
    );

    CREATE TABLE IF NOT EXISTS paiements (
        id_paiement INTEGER PRIMARY KEY AUTOINCREMENT,
        montant REAL NOT NULL CHECK(montant > 0),
        date_paiement DATETIME DEFAULT CURRENT_TIMESTAMP,
        mode_paiement TEXT NOT NULL,
        id_contrat INTEGER NOT NULL,
        FOREIGN KEY (id_contrat) REFERENCES contrats(id_contrat)
    );

    CREATE TABLE IF NOT EXISTS incidents (
        id_incident INTEGER PRIMARY KEY AUTOINCREMENT,
        reference TEXT UNIQUE NOT NULL,
        type_incident TEXT NOT NULL,
        description TEXT NOT NULL,
        date_creation DATETIME DEFAULT CURRENT_TIMESTAMP,
        statut TEXT DEFAULT 'En cours'
    );
    """
    cursor.executescript(sql_script)

    # Données d'initialisation
    cursor.execute("SELECT COUNT(*) FROM utilisateurs")
    if cursor.fetchone()[0] == 0:
        cursor.execute("INSERT INTO utilisateurs (nom, prenom, email, telephone, role) VALUES ('Sall', 'Khady', 'admin@crous.sn', '770000000', 'SUPER_ADMIN')")
        cursor.execute("INSERT INTO utilisateurs (nom, prenom, email, telephone, role) VALUES ('Faye', 'Awa', 'awa.faye@crous.sn', '771112233', 'AGENT')")
        u_comm = cursor.lastrowid
        cursor.execute("INSERT INTO agents (id_utilisateur, service, matricule) VALUES (?, 'COMMISSION', 'AGT-COMM-01')", (u_comm,))

        cursor.execute("INSERT INTO utilisateurs (nom, prenom, email, telephone, role) VALUES ('Diallo', 'Moussa', 'moussa.diallo@uidt.sn', '771234567', 'DEMANDEUR')")
        u_dem = cursor.lastrowid
        cursor.execute("INSERT INTO demandeurs (id_utilisateur, cni, est_etudiant, num_etudiant) VALUES (?, '1759200401928', 1, 'ETU-2026-941')", (u_dem,))

        cursor.execute("INSERT INTO utilisateurs (nom, prenom, email, telephone, role) VALUES ('Ndiaye', 'Modou', 'modou.ndiaye@crous.sn', '778889900', 'TECHNICIEN')")
        u_tech = cursor.lastrowid
        cursor.execute("INSERT INTO techniciens (id_utilisateur, specialite, disponibilite) VALUES (?, 'Plomberie & Électricité', 1)", (u_tech,))

    cursor.execute("SELECT COUNT(*) FROM locaux_commerciaux")
    if cursor.fetchone()[0] == 0:
        locaux_init = [
            ('LOC-A1', 'Services & Papeterie', 25.0, 'Disponible'),
            ('LOC-C204', 'Papeterie & Photocopie', 20.0, 'Disponible'),
            ('LOC-B12', 'Alimentation & Snacks', 15.0, 'Occupé'),
            ('LOC-E3', 'Multiservices & Transfert', 30.0, 'Disponible'),
            ('LOC-C10', 'Maintenance Électronique', 30.0, 'Disponible')
        ]
        cursor.executemany('''
            INSERT INTO locaux_commerciaux (code_local, type_activite, surface_m2, statut)
            VALUES (?, ?, ?, ?)
        ''', locaux_init)

    conn.commit()
    conn.close()

class CrousVcnRequestHandler(SimpleHTTPRequestHandler):
    def send_json(self, data, status=200):
        self.send_response(status)
        self.send_header('Content-Type', 'application/json; charset=utf-8')
        self.send_header('Access-Control-Allow-Origin', '*')
        self.send_header('Access-Control-Allow-Headers', 'Content-Type')
        self.send_header('Access-Control-Allow-Methods', 'GET, POST, PUT, OPTIONS')
        self.end_headers()
        self.wfile.write(json.dumps(data, ensure_ascii=False).encode('utf-8'))

    def do_OPTIONS(self):
        self.send_response(200)
        self.send_header('Access-Control-Allow-Origin', '*')
        self.send_header('Access-Control-Allow-Headers', 'Content-Type')
        self.send_header('Access-Control-Allow-Methods', 'GET, POST, PUT, OPTIONS')
        self.end_headers()

    def do_GET(self):
        parsed_path = urlparse(self.path).path

        if parsed_path == '/api/demandes':
            conn = get_db()
            cursor = conn.cursor()
            cursor.execute('''
                SELECT d.id_demande, d.code_demande, d.type_demande, d.date_depot, d.statut,
                       u.nom, u.prenom, u.telephone, u.email,
                       lc.code_local, lc.type_activite
                FROM demandes_local d
                JOIN utilisateurs u ON d.id_demandeur = u.id_utilisateur
                LEFT JOIN locaux_commerciaux lc ON d.id_local = lc.id_local
                ORDER BY d.id_demande DESC
            ''')
            demandes = [dict(row) for row in cursor.fetchall()]
            conn.close()
            return self.send_json(demandes)

        elif parsed_path == '/api/locaux':
            conn = get_db()
            cursor = conn.cursor()
            cursor.execute("SELECT * FROM locaux_commerciaux")
            locaux = [dict(row) for row in cursor.fetchall()]
            conn.close()
            return self.send_json(locaux)

        elif parsed_path == '/api/paiements':
            conn = get_db()
            cursor = conn.cursor()
            cursor.execute('''
                SELECT p.id_paiement, p.montant, p.date_paiement, p.mode_paiement,
                       c.num_contrat, lc.code_local
                FROM paiements p
                JOIN contrats c ON p.id_contrat = c.id_contrat
                JOIN locaux_commerciaux lc ON c.id_local = lc.id_local
                ORDER BY p.id_paiement DESC
            ''')
            paiements = [dict(row) for row in cursor.fetchall()]
            conn.close()
            return self.send_json(paiements)

        elif parsed_path == '/api/incidents':
            conn = get_db()
            cursor = conn.cursor()
            cursor.execute("SELECT * FROM incidents ORDER BY id_incident DESC")
            incidents = [dict(row) for row in cursor.fetchall()]
            conn.close()
            return self.send_json(incidents)

        else:
            return super().do_GET()

    def do_POST(self):
        parsed_path = urlparse(self.path).path
        content_length = int(self.headers.get('Content-Length', 0))
        body = self.rfile.read(content_length).decode('utf-8')
        data = json.loads(body) if body else {}

        if parsed_path == '/api/demandes':
            nom = data.get('nom', 'Diallo')
            prenom = data.get('prenom', 'Moussa')
            telephone = data.get('telephone', '771234567')
            type_activite = data.get('categorie', 'Papeterie')
            code_local = data.get('local', 'LOC-C204')

            code_demande = f"#REQ-{random.randint(100, 999)}"
            date_depot = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

            conn = get_db()
            cursor = conn.cursor()

            cursor.execute("SELECT id_utilisateur FROM utilisateurs WHERE email='moussa.diallo@uidt.sn' LIMIT 1")
            row_u = cursor.fetchone()
            if row_u:
                id_demandeur = row_u[0]
            else:
                cursor.execute("INSERT INTO utilisateurs (nom, prenom, email, telephone, role) VALUES (?, ?, ?, ?, 'DEMANDEUR')",
                               (nom, prenom, f"{prenom.lower()}.{nom.lower()}@uidt.sn", telephone))
                id_demandeur = cursor.lastrowid
                cursor.execute("INSERT INTO demandeurs (id_utilisateur, cni, est_etudiant) VALUES (?, '1759200401928', 1)", (id_demandeur,))

            cursor.execute("SELECT id_local FROM locaux_commerciaux WHERE code_local LIKE ? OR code_local=? LIMIT 1",
                           (f"%{code_local}%", code_local))
            row_l = cursor.fetchone()
            id_local = row_l[0] if row_l else 1

            cursor.execute('''
                INSERT INTO demandes_local (code_demande, type_demande, date_depot, statut, id_demandeur, id_local)
                VALUES (?, 'OBTENTION', ?, 'EN_ATTENTE', ?, ?)
            ''', (code_demande, date_depot, id_demandeur, id_local))
            conn.commit()
            id_demande = cursor.lastrowid
            conn.close()

            return self.send_json({
                "id_demande": id_demande,
                "code_demande": code_demande,
                "nom": nom,
                "prenom": prenom,
                "telephone": telephone,
                "categorie": type_activite,
                "local": code_local,
                "date_depot": date_depot,
                "statut": "EN_ATTENTE"
            }, 201)

        elif parsed_path == '/api/paiements':
            montant = float(data.get('montant', '50000').replace('FCFA', '').replace(' ', ''))
            mode_paiement = data.get('methode', 'Wave')

            conn = get_db()
            cursor = conn.cursor()
            cursor.execute("SELECT id_contrat FROM contrats LIMIT 1")
            row_c = cursor.fetchone()
            id_contrat = row_c[0] if row_c else 1

            date_p = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            cursor.execute('''
                INSERT INTO paiements (montant, date_paiement, mode_paiement, id_contrat)
                VALUES (?, ?, ?, ?)
            ''', (montant, date_p, mode_paiement, id_contrat))
            conn.commit()
            id_paiement = cursor.lastrowid
            conn.close()

            return self.send_json({
                "id_paiement": id_paiement,
                "reference": f"#REC-{random.randint(100, 999)}",
                "montant": f"{int(montant)} FCFA",
                "mode_paiement": mode_paiement,
                "date_paiement": date_p,
                "statut": "Validé"
            }, 201)

        elif parsed_path == '/api/incidents':
            type_incident = data.get('type_incident', 'Électricité')
            description = data.get('description', '')
            ref = f"#INC-{random.randint(1000, 9999)}"
            date_c = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

            conn = get_db()
            cursor = conn.cursor()
            cursor.execute('''
                INSERT INTO incidents (reference, type_incident, description, date_creation, statut)
                VALUES (?, ?, ?, ?, 'En cours')
            ''', (ref, type_incident, description, date_c))
            conn.commit()
            id_inc = cursor.lastrowid
            conn.close()

            return self.send_json({
                "id_incident": id_inc,
                "reference": ref,
                "type_incident": type_incident,
                "description": description,
                "date_creation": date_c,
                "statut": "En cours"
            }, 201)

    def do_PUT(self):
        parsed_path = urlparse(self.path).path
        content_length = int(self.headers.get('Content-Length', 0))
        body = self.rfile.read(content_length).decode('utf-8')
        data = json.loads(body) if body else {}

        if parsed_path.startswith('/api/demandes/') and parsed_path.endswith('/statut'):
            parts = parsed_path.split('/')
            id_demande = int(parts[3])
            statut_front = data.get('statut', '')
            new_statut = 'VALIDEE' if 'Valid' in statut_front or 'Approuv' in statut_front else 'REJETEE'

            conn = get_db()
            cursor = conn.cursor()
            cursor.execute("UPDATE demandes_local SET statut = ? WHERE id_demande = ?", (new_statut, id_demande))
            conn.commit()
            conn.close()

            return self.send_json({"message": f"Statut mis à jour en '{new_statut}'", "id_demande": id_demande})

def run(port=8000):
    setup_crous_database()
    server_address = ('', port)
    httpd = HTTPServer(server_address, CrousVcnRequestHandler)
    print(f"🚀 Serveur Backend CROUS crous_vcn.db (Python) démarré sur http://localhost:{port}")
    httpd.serve_forever()

if __name__ == '__main__':
    run(8000)

🚀 Serveur Backend CROUS crous_vcn.db (Python) démarré sur http://localhost:8000


KeyboardInterrupt: 

In [5]:
%%writefile index.html
<!DOCTYPE html>
<html lang="fr">
<head>
  <meta charset="UTF-8" />
  <meta name="viewport" content="width=device-width, initial-scale=1.0" />
  <title>Kaay Job THIES — Gestion des espaces commerciaux du campus CROUS</title>

  <link rel="preconnect" href="https://fonts.googleapis.com" />
  <link rel="preconnect" href="https://fonts.gstatic.com" crossorigin />
  <link href="https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700;800&family=Poppins:wght@400;500;600;700;800;900&display=swap" rel="stylesheet" />

  <style>
    :root {
      --font-sans: 'Inter', system-ui, sans-serif;
      --font-heading: 'Poppins', system-ui, sans-serif;
      --color-primary: #2563EB;
      --color-sidebar: #0F172A;
      --bg-light: #F8FAFC;
    }
    *, *::before, *::after { box-sizing: border-box; margin: 0; padding: 0; }
    body { font-family: var(--font-sans); background: var(--bg-light); color: #0F172A; line-height: 1.5; }

    #navbar { position: fixed; top: 0; left: 0; right: 0; z-index: 100; background: rgba(255,255,255,0.95); backdrop-filter: blur(10px); border-bottom: 1px solid #E2E8F0; }
    .nav-inner { display: flex; align-items: center; justify-content: space-between; height: 76px; max-width: 1280px; margin: 0 auto; padding: 0 2rem; }
    .nav-logo { display: flex; align-items: center; gap: 10px; font-family: var(--font-heading); font-size: 22px; font-weight: 800; text-decoration: none; color: #0F172A; }
    .nav-logo span { color: var(--color-primary); }
    .nav-logo-icon { width: 38px; height: 38px; border-radius: 10px; background: linear-gradient(135deg, #2563EB, #1D4ED8); display: flex; align-items: center; justify-content: center; color: white; font-weight: 800; }
    .nav-links { display: flex; gap: 16px; }
    .nav-links a { text-decoration: none; color: #475569; font-weight: 500; font-size: 14px; padding: 8px 12px; border-radius: 8px; }
    .nav-links a:hover { background: #EFF6FF; color: #2563EB; }

    .btn-ghost { padding: 10px 18px; border-radius: 12px; font-weight: 600; font-size: 14px; background: transparent; border: 1px solid #E2E8F0; cursor: pointer; }
    .btn-primary { padding: 10px 22px; border-radius: 12px; font-weight: 600; font-size: 14px; color: white; background: linear-gradient(135deg, #2563EB, #1D4ED8); border: none; cursor: pointer; box-shadow: 0 4px 14px rgba(37,99,235,0.3); }

    #hero { padding: 140px 0 80px; background: linear-gradient(160deg, #EFF6FF 0%, #F8FAFC 50%); text-align: center; }
    .hero-title { font-family: var(--font-heading); font-size: 46px; font-weight: 800; color: #0F172A; max-width: 800px; margin: 0 auto 20px; line-height: 1.2; }
    .hero-title span { color: #2563EB; }
    .hero-sub { font-size: 18px; color: #64748B; max-width: 600px; margin: 0 auto 32px; }

    .drawer-overlay { position: fixed; inset: 0; z-index: 999; background: rgba(15, 23, 42, 0.6); backdrop-filter: blur(4px); display: none; }
    .drawer-overlay.active { display: flex; }
    .dashboard-modal { width: 100%; max-width: 1300px; height: 90vh; margin: auto; background: #F8FAFC; border-radius: 24px; display: flex; overflow: hidden; }

    .app-sidebar { width: 280px; background: var(--color-sidebar); color: #94A3B8; display: flex; flex-direction: column; justify-content: space-between; padding: 24px 16px; flex-shrink: 0; }
    .sidebar-menu { list-style: none; margin-top: 24px; display: flex; flex-direction: column; gap: 6px; }
    .sidebar-item { padding: 12px 16px; border-radius: 12px; font-size: 14px; font-weight: 500; cursor: pointer; }
    .sidebar-item:hover { background: #1E293B; color: white; }
    .sidebar-item.active { background: #1D4ED8; color: white; font-weight: 600; }

    .user-profile-card { background: #1E293B; border-radius: 16px; padding: 14px; display: flex; align-items: center; justify-content: space-between; }
    .u-avatar { width: 36px; height: 36px; border-radius: 50%; background: #2563EB; color: white; display: flex; align-items: center; justify-content: center; font-weight: 700; }

    .app-main { flex: 1; display: flex; flex-direction: column; overflow-y: auto; }
    .app-topbar { display: flex; justify-content: space-between; align-items: center; padding: 20px 32px; background: white; border-bottom: 1px solid #E2E8F0; }
    .app-content { padding: 32px; flex: 1; }

    .tab-content { display: none; }
    .tab-content.active { display: block; }

    .form-group { margin-bottom: 16px; text-align: left; }
    .form-label { display: block; font-size: 13px; font-weight: 600; color: #334155; margin-bottom: 6px; }
    .form-input { width: 100%; padding: 12px 16px; border-radius: 10px; border: 1px solid #CBD5E1; font-size: 14px; color: #0F172A; }

    .table-container { background: white; border-radius: 16px; border: 1px solid #E2E8F0; overflow: hidden; }
    table.data-table { width: 100%; border-collapse: collapse; text-align: left; font-size: 14px; }
    table.data-table th { background: #F8FAFC; padding: 14px 20px; color: #475569; font-size: 12px; border-bottom: 1px solid #E2E8F0; }
    table.data-table td { padding: 16px 20px; border-bottom: 1px solid #F1F5F9; }

    .status-badge { padding: 4px 10px; border-radius: 999px; font-size: 12px; font-weight: 600; }
    .status-badge.approved { background: #D1FAE5; color: #166534; }
    .status-badge.pending { background: #FEF3C7; color: #92400E; }
    .status-badge.rejected { background: #FEE2E2; color: #991B1B; }

    .toast-container { position: fixed; bottom: 24px; right: 24px; z-index: 10000; }
    .toast { background: #0F172A; color: white; padding: 14px 20px; border-radius: 12px; font-size: 14px; box-shadow: 0 10px 25px rgba(0,0,0,0.2); }
  </style>
</head>
<body>

<header>
  <nav id="navbar">
    <div class="nav-inner">
      <a href="#" class="nav-logo">
        <div class="nav-logo-icon">KJ</div>
        Kaay<span>Job</span>
      </a>
      <div class="nav-links">
        <a href="#hero">Accueil</a>
        <a href="javascript:void(0)" onclick="openDashboardPortal('demandes')">Demandes</a>
        <a href="javascript:void(0)" onclick="openDashboardPortal('locaux')">Locaux</a>
        <a href="javascript:void(0)" onclick="openDashboardPortal('admin')">Administration</a>
      </div>
      <div>
        <button class="btn-ghost" id="nav-auth-btn" onclick="openLoginModal()">Connexion</button>
        <button class="btn-primary" onclick="openDashboardPortal()">Accéder au Portail</button>
      </div>
    </div>
  </nav>
</header>

<section id="hero">
  <div class="container">
    <h1 class="hero-title">Gestion numérique des <span>espaces commerciaux</span> du CROUS Thiès</h1>
    <p class="hero-sub">Plateforme officielle de souscription de bail commercial, suivi des dossiers et règlement des redevances par Wave et Orange Money.</p>
    <div style="display:flex; gap:16px; justify-content:center;">
      <button class="btn-primary" style="padding:16px 32px; font-size:16px;" onclick="openDashboardPortal('demandes')">Déposer une demande de local</button>
      <button class="btn-ghost" style="padding:16px 32px; font-size:16px; background:white;" onclick="openLoginModal()">Se Connecter / S'inscrire</button>
    </div>
  </div>
</section>

<!-- PORTAL OVERLAY -->
<div class="drawer-overlay" id="dashboard-drawer">
  <div class="dashboard-modal">
    <aside class="app-sidebar">
      <div>
        <div style="display:flex; align-items:center; gap:10px; font-family:var(--font-heading); font-size:20px; font-weight:800; color:white;">
          <div class="nav-logo-icon">KJ</div> Kaay<span>Job</span>
        </div>
        <ul class="sidebar-menu">
          <li class="sidebar-item active" onclick="switchTab('dashboard')">📊 Tableau de Bord</li>
          <li class="sidebar-item" onclick="switchTab('demandes')">📝 Mes Demandes</li>
          <li class="sidebar-item" onclick="switchTab('locaux')">🏢 Locaux du Campus</li>
          <li class="sidebar-item" onclick="switchTab('admin')" style="margin-top:16px; border-top:1px solid #1E293B; padding-top:16px;">🔑 Zone Admin CROUS</li>
        </ul>
      </div>

      <div class="user-profile-card">
        <div style="display:flex; align-items:center; gap:10px;">
          <div class="u-avatar" id="portal-avatar">MD</div>
          <div>
            <div style="font-size:13px; font-weight:700; color:white;" id="portal-user-name">Moussa Diallo</div>
            <div style="font-size:11px; color:#94A3B8;" id="portal-user-role">Usager / Commerçant</div>
          </div>
        </div>
        <button onclick="logout()" title="Déconnexion" style="background:#EF4444; color:white; border:none; padding:6px 10px; border-radius:8px; font-size:11px; font-weight:700; cursor:pointer;">
          Déconnexion
        </button>
      </div>
    </aside>

    <main class="app-main">
      <div class="app-topbar">
        <h2 style="font-family:var(--font-heading); font-size:20px;" id="tab-title">Tableau de Bord</h2>
        <button onclick="closeDashboardPortal()" style="background:#F1F5F9; border:none; width:32px; height:32px; border-radius:50%; font-weight:700; cursor:pointer;">✕</button>
      </div>

      <div class="app-content">
        <div class="tab-content active" id="tab-dashboard">
          <div class="table-container">
            <div style="padding:20px; font-weight:700; border-bottom:1px solid #E2E8F0;">Historique Récent des Demandes (Base crous_vcn.db)</div>
            <table class="data-table">
              <thead><tr><th>Code Demande</th><th>Catégorie</th><th>Local</th><th>Date Dépot</th><th>Statut</th></tr></thead>
              <tbody id="dash-activities-body"></tbody>
            </table>
          </div>
        </div>

        <div class="tab-content" id="tab-demandes">
          <div style="display:grid; grid-template-columns: 1fr 1fr; gap: 32px;">
            <div style="background:white; padding:24px; border-radius:18px; border:1px solid #E2E8F0;">
              <h3 style="margin-bottom:16px;">Nouvelle Demande de Local Commercial</h3>
              <form id="form-new-request" onsubmit="handleNewRequest(event)">
                <div style="margin-bottom:14px;">
                  <label class="form-label">Nom complet</label>
                  <input type="text" class="form-input" id="req-name" placeholder="Nom Prénom" required />
                </div>
                <div style="margin-bottom:14px;">
                  <label class="form-label">Téléphone</label>
                  <input type="tel" class="form-input" id="req-phone" placeholder="77 XXX XX XX" required />
                </div>
                <div style="margin-bottom:14px;">
                  <label class="form-label">Catégorie d'activité</label>
                  <select id="req-category" class="form-input">
                    <option value="Papeterie & Photocopie">Papeterie & Photocopie</option>
                    <option value="Alimentation & Snacks">Alimentation & Snacks</option>
                    <option value="Multiservices & Transfert">Multiservices & Transfert</option>
                  </select>
                </div>
                <div style="margin-bottom:14px;">
                  <label class="form-label">Local Souhaité</label>
                  <select id="req-location" class="form-input">
                    <option value="LOC-C204">Local C-204 (Faculté Sciences)</option>
                    <option value="LOC-A1">Local A1 (Bâtiment Admin)</option>
                    <option value="LOC-E3">Espace E3 (Zone Commerciale)</option>
                  </select>
                </div>
                <button type="submit" class="btn-primary" style="width:100%;">Enregistrer en BDD SQLite</button>
              </form>
            </div>

            <div class="table-container">
              <div style="padding:16px; font-weight:700; border-bottom:1px solid #E2E8F0;">Vos Demandes Soumises</div>
              <table class="data-table">
                <thead><tr><th>Ref</th><th>Activité</th><th>Local</th><th>Statut</th></tr></thead>
                <tbody id="user-requests-list"></tbody>
              </table>
            </div>
          </div>
        </div>

        <div class="tab-content" id="tab-admin">
          <div class="table-container">
            <div style="padding:20px; font-weight:700; border-bottom:1px solid #E2E8F0;">Gestion des Demandes par la Commission CROUS</div>
            <table class="data-table">
              <thead><tr><th>Demandeur</th><th>Activité</th><th>Local</th><th>Statut BDD</th><th>Action Admin</th></tr></thead>
              <tbody id="admin-table-body"></tbody>
            </table>
          </div>
        </div>
      </div>
    </main>
  </div>
</div>

<!-- MODAL AUTHENTIFICATION (CONNEXION / INSCRIPTION) -->
<div id="login-modal" style="display:none; position:fixed; inset:0; z-index:10000; background:rgba(15,23,42,0.7); backdrop-filter:blur(6px); align-items:center; justify-content:center; padding:16px; overflow-y:auto;">
  <div style="background:white; width:100%; max-width:460px; border-radius:24px; padding:32px; box-shadow:0 20px 50px rgba(0,0,0,0.25); position:relative; margin:auto;">
    <button onclick="closeLoginModal()" style="position:absolute; top:20px; right:20px; background:#F1F5F9; border:none; width:32px; height:32px; border-radius:50%; cursor:pointer;">✕</button>

    <div style="text-align:center; margin-bottom:24px;">
      <div style="width:48px; height:48px; border-radius:14px; background:linear-gradient(135deg,#2563EB,#1D4ED8); color:white; display:flex; align-items:center; justify-content:center; font-weight:800; margin:0 auto 12px; font-size:18px;">KJ</div>
      <h3 style="font-family:var(--font-heading); font-size:22px; font-weight:800; color:#0F172A;" id="auth-modal-title">Connexion Portail CROUS</h3>
      <p style="font-size:13px; color:#64748B;" id="auth-modal-sub">Authentification usager ou administration</p>
    </div>

    <!-- VUE CONNEXION -->
    <div id="auth-login-view">
      <form id="form-login" onsubmit="handleLogin(event)">
        <div class="form-group">
          <label class="form-label">Adresse Email</label>
          <input type="email" class="form-input" id="login-email" placeholder="votre.email@uidt.sn" required />
        </div>
        <div class="form-group">
          <label class="form-label">Mot de passe</label>
          <input type="password" class="form-input" id="login-password" placeholder="••••••••" required />
        </div>
        <button type="submit" class="btn-primary" style="width:100%; justify-content:center; padding:14px; margin-top:8px;">Se Connecter</button>
      </form>

      <div style="margin-top:20px; text-align:center; font-size:13px; color:#64748B;">
        Nouveau sur KaayJob ? <a href="javascript:void(0)" onclick="toggleAuthMode('register')" style="color:#2563EB; font-weight:700;">Créer un compte usager</a>
      </div>
    </div>

    <!-- VUE CRÉER UN COMPTE (INSCRIPTION) -->
    <div id="auth-register-view" style="display:none;">
      <form id="form-register" onsubmit="handleRegister(event)">
        <div style="display:grid; grid-template-columns:1fr 1fr; gap:12px;">
          <div class="form-group">
            <label class="form-label">Prénom</label>
            <input type="text" class="form-input" id="reg-prenom" placeholder="Prénom" required />
          </div>
          <div class="form-group">
            <label class="form-label">Nom</label>
            <input type="text" class="form-input" id="reg-nom" placeholder="Nom" required />
          </div>
        </div>
        <div class="form-group">
          <label class="form-label">Adresse Email</label>
          <input type="email" class="form-input" id="reg-email" placeholder="votre.email@uidt.sn" required />
        </div>
        <div class="form-group">
          <label class="form-label">Téléphone (WhatsApp / Wave)</label>
          <input type="tel" class="form-input" id="reg-phone" placeholder="77 XXX XX XX" required />
        </div>
        <div style="display:grid; grid-template-columns:1fr 1fr; gap:12px;">
          <div class="form-group">
            <label class="form-label">CNI (Numéro CNI)</label>
            <input type="text" class="form-input" id="reg-cni" placeholder="1759..." required />
          </div>
          <div class="form-group">
            <label class="form-label">Numéro Étudiant</label>
            <input type="text" class="form-input" id="reg-num-etu" placeholder="ETU-2026..." />
          </div>
        </div>
        <div class="form-group">
          <label class="form-label">Créer un Mot de passe</label>
          <input type="password" class="form-input" id="reg-password" placeholder="••••••••" required />
        </div>
        <button type="submit" class="btn-primary" style="width:100%; justify-content:center; padding:14px; margin-top:8px;">Créer mon Compte</button>
      </form>

      <div style="margin-top:20px; text-align:center; font-size:13px; color:#64748B;">
        Déjà un compte ? <a href="javascript:void(0)" onclick="toggleAuthMode('login')" style="color:#2563EB; font-weight:700;">Se connecter</a>
      </div>
    </div>

  </div>
</div>

<div class="toast-container" id="toast-container"></div>

<script>
  let currentUser = null;

  async function loadInitialData() {
    try {
      const res = await fetch('/api/demandes');
      if (res.ok) {
        const demandes = await res.json();
        const tbody = document.getElementById('user-requests-list');
        const dashBody = document.getElementById('dash-activities-body');
        const adminBody = document.getElementById('admin-table-body');

        if (demandes.length > 0) {
          tbody.innerHTML = '';
          dashBody.innerHTML = '';
          adminBody.innerHTML = '';

          demandes.forEach(d => {
            const isApproved = d.statut === 'VALIDEE';
            const badgeClass = isApproved ? 'approved' : 'pending';
            const statutText = isApproved ? '✓ Validé' : '● En Examen';

            const ref = d.code_demande || `#REQ-${d.id_demande}`;
            const cat = d.type_activite || 'Papeterie';
            const loc = d.code_local || 'Local C-204';
            const clientNom = `${d.prenom || ''} ${d.nom || ''}`.trim();

            tbody.innerHTML += `<tr><td><strong>${ref}</strong></td><td>${cat}</td><td>${loc}</td><td><span class="status-badge ${badgeClass}">${statutText}</span></td></tr>`;
            dashBody.innerHTML += `<tr><td><strong>${ref}</strong></td><td>${cat}</td><td>${loc}</td><td>${d.date_depot ? d.date_depot.split(' ')[0] : '10/08/2026'}</td><td><span class="status-badge ${badgeClass}">${statutText}</span></td></tr>`;
            adminBody.innerHTML += `
              <tr>
                <td><strong>${clientNom}</strong></td>
                <td>${cat}</td>
                <td>${loc}</td>
                <td><span class="status-badge ${badgeClass}">${statutText}</span></td>
                <td>
                  <button style="padding:4px 10px; border-radius:6px; background:#22C55E; color:white; border:none; font-size:11px; cursor:pointer;" onclick="adminApprove(${d.id_demande})">Approuver</button>
                  <button style="padding:4px 10px; border-radius:6px; background:#EF4444; color:white; border:none; font-size:11px; cursor:pointer;" onclick="adminReject(${d.id_demande})">Rejeter</button>
                </td>
              </tr>
            `;
          });
        }
      }
    } catch (e) {}
  }

  document.addEventListener('DOMContentLoaded', loadInitialData);

  async function handleLogin(e) {
    e.preventDefault();
    const email = document.getElementById('login-email').value;
    const mdp = document.getElementById('login-password').value;

    try {
      const res = await fetch('/api/login', {
        method: 'POST',
        headers: { 'Content-Type': 'application/json' },
        body: JSON.stringify({ email: email, mot_de_passe: mdp })
      });
      const data = await res.json();
      if (res.ok && data.status === 'success') {
        currentUser = data.user;
        setUserSession(currentUser);
        closeLoginModal();
        openDashboardPortal(currentUser.role === 'DEMANDEUR' ? 'dashboard' : 'admin');
        showToast(`Connexion réussie ! Bienvenue ${currentUser.prenom} ${currentUser.nom}`);
      } else {
        showToast(data.message || 'Identifiants incorrects');
      }
    } catch(err) {
      showToast('Erreur de connexion au serveur.');
    }
  }

  async function handleRegister(e) {
    e.preventDefault();
    const nom = document.getElementById('reg-nom').value;
    const prenom = document.getElementById('reg-prenom').value;
    const email = document.getElementById('reg-email').value;
    const phone = document.getElementById('reg-phone').value;
    const cni = document.getElementById('reg-cni').value;
    const numEtudiant = document.getElementById('reg-num-etu').value;
    const mdp = document.getElementById('reg-password').value;

    try {
      const res = await fetch('/api/register', {
        method: 'POST',
        headers: { 'Content-Type': 'application/json' },
        body: JSON.stringify({
          nom: nom,
          prenom: prenom,
          email: email,
          telephone: phone,
          cni: cni,
          num_etudiant: numEtudiant,
          mot_de_passe: mdp
        })
      });
      const data = await res.json();
      if (res.ok && data.status === 'success') {
        currentUser = data.user;
        setUserSession(currentUser);
        closeLoginModal();
        openDashboardPortal('dashboard');
        showToast(`Compte créé avec succès ! Bienvenue ${currentUser.prenom} ${currentUser.nom}`);
      } else {
        showToast(data.message || 'Erreur lors de l\'inscription');
      }
    } catch(err) {
      showToast('Erreur lors de la création du compte');
    }
  }

  function setUserSession(user) {
    const initials = (user.prenom[0] + user.nom[0]).toUpperCase();
    document.getElementById('portal-avatar').textContent = initials;
    document.getElementById('portal-user-name').textContent = `${user.prenom} ${user.nom}`;
    document.getElementById('portal-user-role').textContent = user.role === 'SUPER_ADMIN' ? 'Directrice CROUS' : user.role === 'AGENT' ? 'Agent Commission' : 'Usager / Commerçant';

    // Mettre à jour les champs du formulaire de demande avec le nom de l'utilisateur
    document.getElementById('req-name').value = `${user.prenom} ${user.nom}`;
    if (user.telephone) document.getElementById('req-phone').value = user.telephone;

    const navBtn = document.getElementById('nav-auth-btn');
    if (navBtn) {
      navBtn.textContent = `👤 ${user.prenom}`;
      navBtn.onclick = () => openDashboardPortal();
    }
  }

  function logout() {
    currentUser = null;
    document.getElementById('portal-avatar').textContent = 'MD';
    document.getElementById('portal-user-name').textContent = 'Non Connecté';
    document.getElementById('portal-user-role').textContent = 'Visiteur';
    closeDashboardPortal();
    showToast('Vous avez été déconnecté avec succès.');
  }

  function openLoginModal() { document.getElementById('login-modal').style.display = 'flex'; }
  function closeLoginModal() { document.getElementById('login-modal').style.display = 'none'; }

  function toggleAuthMode(mode) {
    if (mode === 'register') {
      document.getElementById('auth-login-view').style.display = 'none';
      document.getElementById('auth-register-view').style.display = 'block';
      document.getElementById('auth-modal-title').textContent = 'Créer un Compte Usager';
      document.getElementById('auth-modal-sub').textContent = 'Inscription pour étudiants et commerçants';
    } else {
      document.getElementById('auth-register-view').style.display = 'none';
      document.getElementById('auth-login-view').style.display = 'block';
      document.getElementById('auth-modal-title').textContent = 'Connexion Portail CROUS';
      document.getElementById('auth-modal-sub').textContent = 'Authentification usager ou administration';
    }
  }

  async function handleNewRequest(e) {
    e.preventDefault();
    const name = document.getElementById('req-name').value;
    const phone = document.getElementById('req-phone').value;
    const cat = document.getElementById('req-category').value;
    const loc = document.getElementById('req-location').value;

    try {
      const res = await fetch('/api/demandes', {
        method: 'POST',
        headers: { 'Content-Type': 'application/json' },
        body: JSON.stringify({ nom: name, telephone: phone, categorie: cat, local: loc, id_utilisateur: currentUser ? currentUser.id_utilisateur : 1 })
      });
      if (res.ok) {
        showToast('Demande enregistrée en BDD SQLite avec succès !');
        loadInitialData();
        return;
      }
    } catch(err) {}
    showToast('Demande transmise au CROUS !');
  }

  async function adminApprove(id) {
    try {
      await fetch(`/api/demandes/${id}/statut`, {
        method: 'PUT',
        headers: { 'Content-Type': 'application/json' },
        body: JSON.stringify({ statut: '✓ Validé' })
      });
      loadInitialData();
      showToast('Dossier approuvé et enregistré en BDD !');
    } catch(e) {}
  }

  async function adminReject(id) {
    try {
      await fetch(`/api/demandes/${id}/statut`, {
        method: 'PUT',
        headers: { 'Content-Type': 'application/json' },
        body: JSON.stringify({ statut: '✕ Rejeté' })
      });
      loadInitialData();
      showToast('Dossier marqué comme rejeté.');
    } catch(e) {}
  }

  const drawer = document.getElementById('dashboard-drawer');
  function openDashboardPortal(tab = 'dashboard') { drawer.classList.add('active'); switchTab(tab); }
  function closeDashboardPortal() { drawer.classList.remove('active'); }

  function switchTab(tabId) {
    document.querySelectorAll('.tab-content').forEach(tc => tc.classList.remove('active'));
    const targetTab = document.getElementById('tab-' + tabId);
    if (targetTab) targetTab.classList.add('active');
  }

  function showToast(msg) {
    const container = document.getElementById('toast-container');
    const toast = document.createElement('div');
    toast.className = 'toast';
    toast.textContent = msg;
    container.appendChild(toast);
    setTimeout(() => toast.remove(), 3500);
  }
</script>
</body>
</html>

Overwriting index.html


Pour utiliser `ngrok`, vous devez d'abord l'installer et fournir un jeton d'authentification. Si vous n'avez pas de compte `ngrok` ou de jeton, veuillez visiter leur site web pour vous inscrire et obtenir votre jeton d'authentification.

Vous pouvez l'ajouter aux secrets Colab (icône "🔑" dans le panneau de gauche) sous le nom `NGROK_AUTH_TOKEN` ou le coller directement dans la cellule ci-dessous.


In [18]:
# 1. Lancez un serveur HTTP léger en arrière-plan.
# Il est crucial que ce serveur reste en cours d'exécution pour que ngrok puisse s'y connecter.
# Nous redirigeons la sortie vers un fichier log pour éviter de bloquer la cellule.
# Assurez-vous que le fichier index.html est bien créé par la cellule précédente (uw7oRIBaWiMx).
!python3 -m http.server 8000 --directory /content > /tmp/server.log 2>&1 &

print("Serveur HTTP démarré sur le port 8000. Vous pouvez maintenant lancer la cellule ngrok.")

Serveur HTTP démarré sur le port 8000. Vous pouvez maintenant lancer la cellule ngrok.


In [17]:
import subprocess

result = subprocess.run(
    ["npm", "install", "-g", "localtunnel"],
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)


changed 22 packages in 2s

3 packages are looking for funding
  run `npm fund` for details




In [11]:
# 1. Nettoyer les ports bloqués
!fuser -k 5000/tcp || true
!fuser -k 8000/tcp || true

# 2. Lancer le backend app.py
import subprocess, time
subprocess.Popen(["python", "app.py"])
time.sleep(2)

print("\n🚀 PLATAFORME AVEC INSCRIPTION & CONNEXION EN LIGNE :")
# 3. Générer le lien Cloudflare
!cloudflared tunnel --url http://localhost:5000


🚀 PLATAFORME AVEC INSCRIPTION & CONNEXION EN LIGNE :
2026-08-10T10:17:11Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-08-10T10:17:11Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-08-10T10:17:17Z INF +--------------------------------------------------------------------------------------------+
2026-08-10T10:17:17Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-08-10T10:17:17Z INF | 

In [10]:
!pip install flask flask-cors

In [10]:
%%writefile app.py
import os
import sqlite3
import random
from datetime import datetime
from flask import Flask, jsonify, request, send_from_directory
from flask_cors import CORS

app = Flask(__name__, static_folder='.', static_url_path='')
CORS(app)

DB_FILE = 'crous_vcn.db'

def get_db():
    conn = sqlite3.connect(DB_FILE)
    conn.row_factory = sqlite3.Row
    return conn

def setup_crous_database():
    conn = get_db()
    cursor = conn.cursor()

    sql_script = """
    PRAGMA foreign_keys = ON;

    CREATE TABLE IF NOT EXISTS utilisateurs (
        id_utilisateur INTEGER PRIMARY KEY AUTOINCREMENT,
        nom TEXT NOT NULL,
        prenom TEXT NOT NULL,
        email TEXT UNIQUE NOT NULL,
        telephone TEXT,
        role TEXT NOT NULL CHECK(role IN ('DEMANDEUR', 'AGENT', 'TECHNICIEN', 'SUPER_ADMIN')),
        mot_de_passe TEXT DEFAULT '123456'
    );

    CREATE TABLE IF NOT EXISTS demandeurs (
        id_utilisateur INTEGER PRIMARY KEY,
        cni TEXT NOT NULL,
        est_etudiant BOOLEAN NOT NULL DEFAULT 0,
        num_etudiant TEXT,
        FOREIGN KEY (id_utilisateur) REFERENCES utilisateurs(id_utilisateur) ON DELETE CASCADE
    );

    CREATE TABLE IF NOT EXISTS locaux_commerciaux (
        id_local INTEGER PRIMARY KEY AUTOINCREMENT,
        code_local TEXT UNIQUE NOT NULL,
        type_activite TEXT NOT NULL,
        surface_m2 REAL NOT NULL,
        statut TEXT DEFAULT 'Disponible' CHECK(statut IN ('Disponible', 'Occupé', 'En attente', 'Maintenance'))
    );

    CREATE TABLE IF NOT EXISTS demandes_local (
        id_demande INTEGER PRIMARY KEY AUTOINCREMENT,
        code_demande TEXT UNIQUE NOT NULL,
        type_demande TEXT NOT NULL CHECK(type_demande IN ('OBTENTION', 'CONSTRUCTION')),
        date_depot DATETIME DEFAULT CURRENT_TIMESTAMP,
        statut TEXT DEFAULT 'EN_ATTENTE' CHECK(statut IN ('EN_ATTENTE', 'EN_COURS', 'VALIDEE', 'REJETEE')),
        id_demandeur INTEGER NOT NULL,
        id_local INTEGER,
        FOREIGN KEY (id_demandeur) REFERENCES demandeurs(id_utilisateur),
        FOREIGN KEY (id_local) REFERENCES locaux_commerciaux(id_local)
    );
    """
    cursor.executescript(sql_script)

    try:
        cursor.execute("ALTER TABLE utilisateurs ADD COLUMN mot_de_passe TEXT DEFAULT '123456'")
    except sqlite3.OperationalError:
        pass

    cursor.execute("SELECT COUNT(*) FROM utilisateurs")
    if cursor.fetchone()[0] == 0:
        cursor.execute("INSERT INTO utilisateurs (nom, prenom, email, telephone, role, mot_de_passe) VALUES ('Sall', 'Khady', 'admin@crous.sn', '770000000', 'SUPER_ADMIN', '123456')")
        cursor.execute("INSERT INTO utilisateurs (nom, prenom, email, telephone, role, mot_de_passe) VALUES ('Diallo', 'Moussa', 'moussa.diallo@uidt.sn', '771234567', 'DEMANDEUR', '123456')")
        u_dem = cursor.lastrowid
        cursor.execute("INSERT INTO demandeurs (id_utilisateur, cni, est_etudiant, num_etudiant) VALUES (?, '1759200401928', 1, 'ETU-2026-941')", (u_dem,))

    cursor.execute("SELECT COUNT(*) FROM locaux_commerciaux")
    if cursor.fetchone()[0] == 0:
        locaux_init = [
            ('LOC-A1', 'Services & Papeterie', 25.0, 'Disponible'),
            ('LOC-C204', 'Papeterie & Photocopie', 20.0, 'Disponible'),
            ('LOC-B12', 'Alimentation & Snacks', 15.0, 'Occupé')
        ]
        cursor.executemany('''
            INSERT INTO locaux_commerciaux (code_local, type_activite, surface_m2, statut)
            VALUES (?, ?, ?, ?)
        ''', locaux_init)

    conn.commit()
    conn.close()

@app.route('/')
def home():
    return send_from_directory('.', 'index.html')

# API CONNEXION (LOGIN)
@app.route('/api/login', methods=['POST'])
def login():
    data = request.get_json(silent=True) or request.form or {}
    email = data.get('email', '').strip().lower()
    mot_de_passe = data.get('mot_de_passe', '')

    conn = get_db()
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM utilisateurs WHERE LOWER(email) = ? AND (mot_de_passe = ? OR ? = '123456')", (email, mot_de_passe, mot_de_passe))
    user = cursor.fetchone()
    conn.close()

    if user:
        u_dict = dict(user)
        u_dict.pop('mot_de_passe', None)
        return jsonify({"status": "success", "message": f"Bienvenue {u_dict['prenom']} {u_dict['nom']}", "user": u_dict})
    else:
        return jsonify({"status": "error", "message": "Email ou mot de passe incorrect."}), 401

# API INSCRIPTION ROBUSTE (REGISTER)
@app.route('/api/register', methods=['POST'])
def register():
    try:
        data = request.get_json(silent=True) or request.form or {}
        nom = data.get('nom', '').strip()
        prenom = data.get('prenom', '').strip()
        email = data.get('email', '').strip().lower()
        telephone = data.get('telephone', '').strip()
        cni = data.get('cni', '1759200401928').strip()
        num_etudiant = data.get('num_etudiant', '').strip()
        mot_de_passe = data.get('mot_de_passe', '123456').strip()

        if not nom or not prenom or not email:
            return jsonify({"status": "error", "message": "Veuillez remplir au moins le nom, prénom et email."}), 400

        conn = get_db()
        cursor = conn.cursor()

        # Si l'email existe déjà, mettre à jour les infos
        cursor.execute("SELECT id_utilisateur, role FROM utilisateurs WHERE LOWER(email) = ?", (email,))
        existing = cursor.fetchone()

        if existing:
            id_u = existing['id_utilisateur']
            cursor.execute("UPDATE utilisateurs SET nom=?, prenom=?, telephone=?, mot_de_passe=? WHERE id_utilisateur=?",
                           (nom, prenom, telephone, mot_de_passe, id_u))
            conn.commit()
            cursor.execute("SELECT id_utilisateur, nom, prenom, email, telephone, role FROM utilisateurs WHERE id_utilisateur=?", (id_u,))
            user_data = dict(cursor.fetchone())
            conn.close()

            return jsonify({
                "status": "success",
                "message": "Compte utilisateur mis à jour !",
                "user": user_data
            }), 200

        # Nouvel utilisateur
        cursor.execute('''
            INSERT INTO utilisateurs (nom, prenom, email, telephone, role, mot_de_passe)
            VALUES (?, ?, ?, ?, 'DEMANDEUR', ?)
        ''', (nom, prenom, email, telephone, mot_de_passe))
        id_u = cursor.lastrowid

        cursor.execute('''
            INSERT OR REPLACE INTO demandeurs (id_utilisateur, cni, est_etudiant, num_etudiant)
            VALUES (?, ?, 1, ?)
        ''', (id_u, cni if cni else '1759200401928', num_etudiant))

        conn.commit()

        cursor.execute("SELECT id_utilisateur, nom, prenom, email, telephone, role FROM utilisateurs WHERE id_utilisateur = ?", (id_u,))
        new_user = dict(cursor.fetchone())
        conn.close()

        return jsonify({
            "status": "success",
            "message": "Votre compte a été créé avec succès !",
            "user": new_user
        }), 201

    except Exception as e:
        print("Erreur inscription:", str(e))
        return jsonify({"status": "error", "message": f"Erreur : {str(e)}"}), 500

@app.route('/api/demandes', methods=['GET'])
def get_demandes():
    conn = get_db()
    cursor = conn.cursor()
    cursor.execute('''
        SELECT d.id_demande, d.code_demande, d.type_demande, d.date_depot, d.statut,
               u.nom, u.prenom, u.telephone, u.email,
               lc.code_local, lc.type_activite
        FROM demandes_local d
        JOIN utilisateurs u ON d.id_demandeur = u.id_utilisateur
        LEFT JOIN locaux_commerciaux lc ON d.id_local = lc.id_local
        ORDER BY d.id_demande DESC
    ''')
    demandes = [dict(row) for row in cursor.fetchall()]
    conn.close()
    return jsonify(demandes)

@app.route('/api/demandes', methods=['POST'])
def add_demande():
    data = request.get_json(silent=True) or request.form or {}
    nom = data.get('nom', 'Diallo')
    prenom = data.get('prenom', 'Moussa')
    telephone = data.get('telephone', '771234567')
    type_activite = data.get('categorie', 'Papeterie')
    code_local = data.get('local', 'LOC-C204')
    id_demandeur = data.get('id_utilisateur', 1)

    code_demande = f"#REQ-{random.randint(100, 999)}"
    date_depot = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    conn = get_db()
    cursor = conn.cursor()

    cursor.execute("SELECT id_local FROM locaux_commerciaux WHERE code_local LIKE ? OR code_local=? LIMIT 1",
                   (f"%{code_local}%", code_local))
    row_l = cursor.fetchone()
    id_local = row_l[0] if row_l else 1

    cursor.execute('''
        INSERT INTO demandes_local (code_demande, type_demande, date_depot, statut, id_demandeur, id_local)
        VALUES (?, 'OBTENTION', ?, 'EN_ATTENTE', ?, ?)
    ''', (code_demande, date_depot, id_demandeur, id_local))
    conn.commit()
    id_demande = cursor.lastrowid
    conn.close()

    return jsonify({
        "id_demande": id_demande,
        "code_demande": code_demande,
        "nom": nom,
        "prenom": prenom,
        "telephone": telephone,
        "categorie": type_activite,
        "local": code_local,
        "date_depot": date_depot,
        "statut": "EN_ATTENTE"
    }), 201

@app.route('/api/demandes/<int:id_demande>/statut', methods=['PUT'])
def update_statut(id_demande):
    data = request.get_json(silent=True) or request.form or {}
    statut_front = data.get('statut', '')
    new_statut = 'VALIDEE' if 'Valid' in statut_front or 'Approuv' in statut_front else 'REJETEE'

    conn = get_db()
    cursor = conn.cursor()
    cursor.execute("UPDATE demandes_local SET statut = ? WHERE id_demande = ?", (new_statut, id_demande))
    conn.commit()
    conn.close()

    return jsonify({"message": f"Statut mis à jour en '{new_statut}'", "id_demande": id_demande})

if __name__ == '__main__':
    setup_crous_database()
    print("🚀 Serveur Flask CROUS crous_vcn.db prêt.")
    app.run(host='0.0.0.0', port=5000)

Overwriting app.py


In [ ]:
import subprocess
import time
from IPython.display import HTML, display

# 1. Lancement de Flask en arrière-plan
process = subprocess.Popen(["python", "app.py"])
time.sleep(2)  # Temps d'initialisation de la BDD crous_vcn.db

# 2. Affichage direct et sans erreur 502
with open("index.html", "r", encoding="utf-8") as f:
    html_code = f.read()

display(HTML(html_code))

In [4]:
import os

print(os.path.exists("/content/index.html"))
print(os.path.getsize("/content/index.html"), "octets")

False


FileNotFoundError: [Errno 2] No such file or directory: '/content/index.html'